# Arctic Ocean

Arctic Ocean hydrography

In [ ]:
# parameters for the cmorized data
caseid          = ''
input_root      = ''
work_root       = ''
www_root        = ''
year_start      = []
year_end        = []

In [ ]:
import yaml
import xarray as xr
import sys
import os
import numpy as np
import xarray as xry
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import matplotlib as mpl
import cartopy.crs as ccrs
import matplotlib.path as mpath
import xesmf as xe
from pathlib import Path
import matplotlib.colors as mcolors
from scipy.spatial import cKDTree
with open("../settings.yml", "r") as f:
    cfg = yaml.safe_load(f)
sys.path.append(cfg["blomster"]["source_root"])
from utils.plots import plot_map2d
from utils import get_grid
from utils.plots import plot_arctic_maps
from utils.analysis import mean_at_depth,load_ocean_grid,deep_ocean_mask,shared_quantile_levels,symmetric_quantile_levels


In [ ]:
# delete before running
# caseid = "n1850GaxgGHG.LM.nor30b25.528.20260928"
# input_root = "/nird/datalake/NS9560K/noresm3/cases"
# work_root = "/scratch/yanchun/diagnostics/blomster"
# www_root = "/nird/datalake/NS9015K/www/diagnostics/blomster/nililiu"
# year_start = 1596
# year_end = 1600

caseid = cfg['experiment']['caseid']
input_root = cfg['experiment']['input_root']
work_root = cfg['experiment']['work_root']
www_root = cfg['experiment']['www_root']
year_start = cfg['experiment']['year_start']
year_end = cfg['experiment']['year_end']


time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
grid_file = cfg['grids']['tnx1v4']['grid_file']
years = range(year_start, year_end+1)
hist_dir = Path(f"{input_root}/{caseid}/ocn/hist")
files = [hist_dir / f"{caseid}.blom.hy.{yr}.nc" for yr in years]
phc_file = cfg['datasets']['PHC3.0']['ANNUAL_file']


## Hovmoller diagrams

### 1.values change with time

In [ ]:
max_depth = 1500
with xr.open_dataset(grid_file) as grid:
    arctic_mask = ((grid["pmask"] > 0)& (grid["plat"] >= 65)& (grid["pdepth"] > 500))
    weights = (grid["parea"].where(arctic_mask, 0).fillna(0).load())
    
temp_profiles = []
salt_profiles = []
for i, file in enumerate(files, 1):
    with xr.open_dataset(file, decode_times=time_coder) as ds:
        temp_one = (ds["templvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
        salt_one = (ds["salnlvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
    temp_profiles.append(temp_one)
    salt_profiles.append(salt_one)
temp_time_depth = xr.concat(temp_profiles, dim="time")
salt_time_depth = xr.concat(salt_profiles, dim="time")

temp_time_levels = shared_quantile_levels([temp_time_depth],nlevels=21)
salt_time_levels = shared_quantile_levels([salt_time_depth],nlevels=21)

In [ ]:
plot_years = temp_time_depth["time"].dt.year.values
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    temp_time_depth["depth"] / 1000,
    temp_time_depth.transpose("depth", "time"),
    cmap="plasma",
    vmin=temp_time_levels[0],
    vmax=temp_time_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Temperature [°C]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    salt_time_depth["depth"] / 1000,
    salt_time_depth.transpose("depth", "time"),
    cmap="plasma",
    vmin=salt_time_levels[0],
    vmax=salt_time_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Salnility [g/kg]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean Salnility",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)
fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

### 2. anomaly change with time

In [ ]:
temp_anomaly = (temp_time_depth- temp_time_depth.mean("time", skipna=True))
salt_anomaly = (salt_time_depth- salt_time_depth.mean("time", skipna=True))
temp_ano_levels = symmetric_quantile_levels(temp_anomaly,nlevels=21)
salt_ano_levels = symmetric_quantile_levels(salt_anomaly,nlevels=21)
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    temp_anomaly["depth"] / 1000,
    temp_anomaly.transpose("depth", "time"),
    cmap="RdBu_r",
    vmin=temp_ano_levels[0],
    vmax=temp_ano_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Temperature [°C]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature anomaly",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    salt_anomaly["depth"] / 1000,
    salt_anomaly.transpose("depth", "time"),
    cmap="RdBu_r",
    vmin=salt_ano_levels[0],
    vmax=salt_ano_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Salnility [g/kg]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean sanility anomaly",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## Vertical 
1. temp profile with observation
2. salt profile with observation

In [ ]:
with xr.open_dataset(grid_file) as grid:
    arctic_mask = ((grid["pmask"] > 0) & (grid["plat"] >= 65) & (grid["pdepth"] > 500))
    weights = (grid["parea"].where(arctic_mask, 0).fillna(0).load())

temp_profiles = []
salt_profiles = []

for i, file in enumerate(files, 1):
    with xr.open_dataset(file, decode_times=time_coder) as ds:
        temp_one = (ds["templvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
        salt_one = (ds["salnlvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
    temp_profiles.append(temp_one)
    salt_profiles.append(salt_one)

temp_profile = (xr.concat(temp_profiles, dim="time").mean("time", skipna=True))
salt_profile = (xr.concat(salt_profiles, dim="time").mean("time", skipna=True))

with xr.open_dataset(phc_file,decode_times=time_coder,) as phc:
    phc_weights = np.cos(np.deg2rad(phc["lat"]))
    phc_weights = phc_weights.where(phc["lat"] >= 65, 0)
    phc_temp_profile = (phc["temp"].sel(depth=slice(0, 4000)).weighted(phc_weights).mean(("lat", "lon"), skipna=True).load())
    phc_salt_profile = (phc["salt"].sel(depth=slice(0, 4000)).weighted(phc_weights).mean(("lat", "lon"), skipna=True).load())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6.5))

ax.plot(temp_profile,temp_profile["depth"] / 1000,color="tab:red",linewidth=2.2,label="NorESM3")

ax.plot(phc_temp_profile,phc_temp_profile["depth"] / 1000,color="black",linestyle="-",linewidth=2,label="PHC3.0")

ax.set_xlabel("Temperature [°C]",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature profile",fontsize=14)

ax.set_ylim(4, 0)
ax.grid(True, alpha=0.3)
ax.legend()
fig.suptitle(f"{caseid}\nYears {year_start}–{year_end}",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6.5))

ax.plot(salt_profile,salt_profile["depth"] / 1000,color="tab:blue",linewidth=2.2,label="NorESM3",)

ax.plot(phc_salt_profile,phc_salt_profile["depth"] / 1000,color="black",linestyle="-",linewidth=2,label="PHC3.0")

ax.set_xlabel("Salinity [g kg$^{-1}$]",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean salinity profile",fontsize=14)

ax.set_ylim(4, 0)
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(f"{caseid}\nYears {year_start}–{year_end}",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Climatology maps
1. surface T & S
2. 400m T & S
3. 1000m T & S
4. 2000m T & S

In [ ]:
# Climatology maps
depths = [0,400,1000,2000]
plat, plon, pclat, pclon , pmask = get_grid.load(grid_file)

temp_model_arctic={}
salt_model_arctic={}
phc_temp_on_model={}
phc_salt_on_model={}
temp_bias={}
salt_bias={}
temp_levels={}
salt_levels={}
temp_bias_levels={}
salt_bias_levels={}
for depth in depths:
    temp_model=mean_at_depth("templvl",files,depth)
    salt_model=mean_at_depth("salnlvl",files,depth)
    ydim, xdim = temp_model.dims[-2:]
    # Arctic
    arctic = xr.DataArray((np.asarray(pmask) > 0) & (np.asarray(plat) >= 60),dims=(ydim, xdim))
    temp_model_arctic[depth] = temp_model.where(arctic)
    salt_model_arctic[depth] = salt_model.where(arctic)
    # interp
    with xr.open_dataset(phc_file, decode_times=time_coder) as ds:
        phc_temp = ds["temp"].interp(depth=depth).load()
        phc_salt = ds["salt"].interp(depth=depth).load()
    # create target grid
    model_grid = xr.Dataset(
        {
            "lon": ((ydim, xdim),np.mod(np.asarray(plon), 360),),
            "lat": ((ydim, xdim),np.asarray(plat),),
        }
    )
    # PHC
    phc_grid = xr.Dataset(coords={"lon": phc_temp["lon"],"lat": phc_temp["lat"],})
    # PHC → model grid
    regridder = xe.Regridder(phc_grid,model_grid,method="bilinear",periodic=True,)
    # skipnan
    phc_temp_on_model[depth] = regridder(phc_temp,skipna=True,na_thres=1.0,).where(arctic)
    phc_salt_on_model[depth] = regridder(phc_salt,skipna=True,na_thres=1.0,).where(arctic)
    # Bias
    temp_bias[depth] = temp_model_arctic[depth]- phc_temp_on_model[depth]
    salt_bias[depth] = salt_model_arctic[depth]- phc_salt_on_model[depth]
    temp_levels[depth] = shared_quantile_levels([temp_model_arctic[depth]],nlevels=21)
    salt_levels[depth] = shared_quantile_levels([salt_model_arctic[depth]],nlevels=21)
    temp_bias_levels[depth] = symmetric_quantile_levels(temp_bias[depth],nlevels=21)
    salt_bias_levels[depth] =symmetric_quantile_levels(salt_bias[depth],nlevels=21)

def plot_model_depth(depth):
    model_settings = [
        (
            temp_model_arctic[depth],
            temp_levels[depth],
            "plasma",
            "Temperature [°C]",
        ),
        (
            salt_model_arctic[depth],
            salt_levels[depth],
            "viridis",
            "Salinity [g kg$^{-1}$]",
        ),
    ]

    model_titles = [
        f"Model Arctic Ocean temperature at {depth} m",
        f"Model Arctic Ocean salinity at {depth} m",
    ]

    fig, axes = plot_arctic_maps(
        lon=plon,
        lat=plat,
        settings=model_settings,
        titles=model_titles,
        suptitle=f"{caseid}\nYears {year_start}–{year_end},depth={depth} m",
        layout='horizontal'
    )

    plt.show()

In [ ]:
plot_model_depth(0)

In [ ]:
plot_model_depth(400)

In [ ]:
plot_model_depth(1000)

In [ ]:
plot_model_depth(2000)

## Bias maps
model_result - PHC3.0_results
1. surface
2. 400m
3. 1000m
4. 2000m

In [ ]:
def plot_bias_depth(depth):
    bias_settings = [
        (
            temp_bias[depth],
            temp_bias_levels[depth],
            "RdBu_r",
            "Temperature bias [°C]",
        ),
        (
            salt_bias[depth],
            salt_bias_levels[depth],
            "RdBu_r",
            "Salinity bias",
        ),
    ]

    bias_titles = [
        f"Temperature bias at {depth} m",
        f"Salinity bias at {depth} m",
    ]

    fig, axes = plot_arctic_maps(
        lon=plon,
        lat=plat,
        settings=bias_settings,
        titles=bias_titles,
        suptitle=(
            f"{caseid}\n"
            f"Years {year_start}–{year_end}, depth = {depth} m"
        ),
        layout='horizontal'
    )

    plt.show()
    

In [ ]:
plot_bias_depth(0)

In [ ]:
plot_bias_depth(400)

In [ ]:
plot_bias_depth(1000)

In [ ]:
plot_bias_depth(2000)

## Transects
Greenland to Svalbard profile
1. temperature section
2. salinity section
3. normal velocity section
4. volume transport
5. heat transport
6. freshwater transport

### basins

In [ ]:
# Transects across the Fram Strait
# recognize diffirent basins
LOCATION = {
    "Baltic Sea": {
        "lat_min": 50.0, "lat_max": 68.0,
        "lon_min": 5.0,  "lon_max": 32.0
    },
    "Barents Sea": {
        "lat_min": 68.0, "lat_max": 84.0,
        "lon_min": 10.0, "lon_max": 65.0
    },
    "Bering Strait": {
        "lat_min": 60.0, "lat_max": 72.0,
        "lon_min": -175.0, "lon_max": -160.0
    },
    "Canada Basin": {
        "lat_min": 68.0, "lat_max": 88.0,
        "lon_min": -170.0, "lon_max": -100.0
    },
    "Chukchi Sea": {
        "lat_min": 64.0, "lat_max": 78.0,
        "lon_min": -180.0, "lon_max": -150.0
    },
    "Eurasian Basin": {
        "lat_min": 80.0, "lat_max": 90.0,
        "lon_min": -20.0, "lon_max": 160.0
    },
    "Fram Strait": {
        "lat_min": 70.0, "lat_max": 85.0,
        "lon_min": -20.0, "lon_max": 25.0
    },
    "Greenland Coast": {
        "lat_min": 58.0, "lat_max": 85.0,
        "lon_min": -70.0, "lon_max": 0.0
    },
    "Irish Sea": {  
        "lat_min": 48.0, "lat_max": 58.0,
        "lon_min": -10.0, "lon_max": 0.0
    },
    "Kara Sea": {
        "lat_min": 68.0, "lat_max": 84.0,
        "lon_min": 50.0, "lon_max": 110.0
    },
    "Makarov Basin": {
        "lat_min": 80.0, "lat_max": 90.0,
        "lon_min": -180.0, "lon_max": -130.0
    },
    "Mediterranean Sea": {
        "lat_min": 28.0, "lat_max": 48.0,
        "lon_min": -10.0, "lon_max": 40.0
    },
    "N Atlantic": {
        "lat_min": -5.0, "lat_max": 75.0,
        "lon_min": -90.0, "lon_max": 25.0
    },
    "Nordic Seas": {
        "lat_min": 58.0, "lat_max": 82.0,
        "lon_min": -30.0, "lon_max": 40.0
    },
    "North Sea": { 
        "lat_min": 48.0, "lat_max": 65.0,
        "lon_min": -8.0, "lon_max": 12.0
    }
}


### transects of T & S

In [ ]:
target_lat = 78.83
target_lons = np.linspace(-20, 15, 36)
max_depth = 1000

def spherical_xyz(lon, lat):
    lon = np.deg2rad(np.asarray(lon))
    lat = np.deg2rad(np.asarray(lat))
    return np.column_stack([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])


with xr.open_dataset(grid_file) as grid_ds:
    plon = grid_ds["plon"].load()
    plat = grid_ds["plat"].load()
    pmask = grid_ds["pmask"].load()
    pdepth = grid_ds["pdepth"].load()

ydim, xdim = plat.dims
model_lon = np.asarray(plon)
model_lat = np.asarray(plat)
model_mask = np.asarray(pmask) > 0
model_bottom = np.asarray(pdepth)

model_lon_180 = (model_lon + 180) % 360 - 180
valid_coordinates = (np.isfinite(model_lon)& np.isfinite(model_lat))
valid_y, valid_x = np.where(valid_coordinates)
tree = cKDTree(spherical_xyz(model_lon[valid_coordinates],model_lat[valid_coordinates]))
target_xyz = spherical_xyz(target_lons,np.full_like(target_lons,target_lat,dtype=float),)
# Find nearest BLOM grid cell
distance, nearest = tree.query(target_xyz)
section_y = valid_y[nearest]
section_x = valid_x[nearest]

section_actual_lon = model_lon[section_y,section_x]
section_actual_lat = model_lat[section_y,section_x]
## selected the nearest grid in grid_file

section_wet = model_mask[section_y,section_x] #bool
section_bottom_raw = model_bottom[section_y,section_x]

# Convert selected longitude into -180 to 180
section_actual_lon_180 = (section_actual_lon + 180) % 360 - 180
section_bottom = np.where(section_wet,section_bottom_raw,0.0)
y_index = xr.DataArray(section_y,dims="section")
x_index = xr.DataArray(section_x,dims="section")

temp_sections = []
salt_sections = []
for i, file in enumerate(files, 1):
    with xr.open_dataset(file,decode_times=time_coder) as ds:
        temp_one = (ds["templvl"].sel(depth=slice(0, max_depth)).isel({ydim: y_index,xdim: x_index}).mean("time").load())
        salt_one = (ds["salnlvl"].sel(depth=slice(0, max_depth)).isel({ ydim: y_index,xdim: x_index}).mean("time").load())
    temp_sections.append(temp_one)
    salt_sections.append(salt_one)
temp_section = (xr.concat(temp_sections,dim="year").mean("year"))
salt_section = (xr.concat(salt_sections,dim="year").mean("year"))

# Assign target longitude as the section coordinate
temp_section = temp_section.assign_coords(longitude=("section",target_lons))
salt_section = salt_section.assign_coords(longitude=("section",target_lons))

# Apply ocean and bathymetry masks
wet_section = xr.DataArray(section_wet,dims="section",coords={"section": temp_section["section"],"longitude": ("section",target_lons),})
bottom = xr.DataArray(section_bottom,dims="section",coords={"section": temp_section["section"],"longitude": ("section",target_lons),})
section_valid = (wet_section& (temp_section["depth"] <= bottom))
temp_section = temp_section.where(section_valid)
salt_section = salt_section.where(section_valid)


In [ ]:
projection = ccrs.NorthPolarStereo(central_longitude=0)
data_crs = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(9, 8),subplot_kw={ "projection": projection,})
# Ocean bathymetry
bathymetry = np.where(model_mask,model_bottom,np.nan)
depth_levels = [0,200,500,1000,2000,3000,4000,5000]
depth_norm = mcolors.BoundaryNorm(depth_levels,ncolors=plt.colormaps["Blues"].N)
bathymetry_plot = ax.pcolormesh(model_lon_180,model_lat,bathymetry,transform=data_crs,cmap="Blues",norm=depth_norm,shading="auto",zorder=1)

# Land
ax.add_feature( cfeature.LAND,facecolor="0.7",edgecolor="black",linewidth=0.5,zorder=3)
ax.coastlines(resolution="50m",linewidth=0.6,zorder=4)

# Ideal target section
ax.plot(target_lons,np.full_like(target_lons,target_lat,dtype=float,),color="red",linewidth=2,transform=data_crs,label=f"Target section: {target_lat:.2f}°N",zorder=6)
# Connect the actual selected grid-cell centres
ax.plot(section_actual_lon_180,section_actual_lat,color="black",linestyle="--",linewidth=0.8,transform=data_crs,label="Actual selected grid path",zorder=7)
# Selected ocean cells
ax.scatter(section_actual_lon_180[section_wet],section_actual_lat[section_wet],s=28,color="black",edgecolor="white",linewidth=0.5,transform=data_crs,label="Selected ocean cells",zorder=8)

# Selected land cells
ax.scatter(section_actual_lon_180[~section_wet],section_actual_lat[~section_wet],s=36,marker="x",color="magenta",linewidth=1.3,transform=data_crs,label="Selected land cells",zorder=9)
# Regional map extent
ax.set_extent([-30, 25, 74, 83],crs=data_crs,)
gridlines = ax.gridlines(crs=data_crs,draw_labels=True,x_inline=False,y_inline=False,xlocs=np.arange(-30, 31, 10),ylocs=np.arange(74, 84, 2),linewidth=0.4,color="0.4",alpha=0.6,linestyle="--",)
gridlines.top_labels = False
gridlines.right_labels = False
ax.set_title("Location of the Fram Strait transect\n"f"{caseid}",fontsize=11,)
fig.colorbar(bathymetry_plot,ax=ax,label="Ocean depth [m]",shrink=0.78,pad=0.05)

ax.legend(loc="lower left",fontsize=8,framealpha=0.9)
plt.show()

In [ ]:
#add T and S 
temperature_contours = [-1.5, -1.0, 0.0, 1.0, 2.0, 2.5]
salinity_contours = [32.0, 33.0, 34.0, 34.5, 34.7, 34.9, 35.0]

fig, axes = plt.subplots(2,1,figsize=(11, 9),sharex=True)
# Temperature
temp_plot = axes[0].pcolormesh(temp_section["longitude"],temp_section["depth"],temp_section,cmap="plasma",shading="auto")
salt_lines_on_temp = axes[0].contour(salt_section["longitude"],salt_section["depth"],salt_section,levels=salinity_contours,colors="black",linewidths=0.7,)
axes[0].clabel(salt_lines_on_temp,inline=True,fontsize=8,fmt=lambda value:f"S={value:.1f}")
axes[0].fill_between(target_lons,section_bottom,max_depth,color="0.4")
axes[0].set_ylim(max_depth , 0)
axes[0].set_ylabel("Depth[m]")
axes[0].set_title("Temperature")
axes[0].grid(alpha=0.2)
fig.colorbar(temp_plot,ax=axes[0],label="Temperature [°C]")

# Salinity
salt_plot = axes[1].pcolormesh(salt_section["longitude"],salt_section["depth"],salt_section,cmap="viridis",shading="auto")
temp_lines_on_salt = axes[1].contour(temp_section["longitude"],temp_section["depth"],temp_section,levels=temperature_contours,colors="black",linewidths=0.7,)
axes[1].clabel(temp_lines_on_salt,inline=True,fontsize=8,fmt=lambda value:f"T={value:.1f}")
axes[1].fill_between(target_lons,section_bottom,max_depth,color="0.4")
axes[1].set_ylim(max_depth,0)
axes[1].set_xlabel("Longitude [°E]")
axes[1].set_ylabel("Depth [m]")
axes[1].set_title("Salinity")
axes[1].grid(alpha=0.2)

fig.colorbar(salt_plot,ax=axes[1],label="Salinity [g kg$^{-1}$]")
fig.suptitle(
    f"{caseid}\n"
    f"Fram Strait section near {target_lat:.2f}°N, "
    f"years {year_start}–{year_end}",
    fontsize=11,
)

fig.subplots_adjust(top=0.89,hspace=0.25)
plt.show()